# 09 - Connect to an LLM API on Azure OpenAI

## Learning objectives
- Set up a connection to Azure OpenAI using the `openai` Python SDK.
- Provide an API key at runtime through an input cell.
- Load a document from `data/pubmed/`.
- Send a simple test prompt to verify the connection.
- Send the loaded document to the model for a short summary.


In [10]:
from pathlib import Path
import getpass
from openai import AzureOpenAI
from openai import APIStatusError


## Step 1 - Enter your Azure OpenAI API key

Run this cell and paste your key when prompted.


In [11]:
subscription_key = getpass.getpass("Enter your Azure OpenAI API key: ")

if not subscription_key.strip():
    raise ValueError("API key is empty. Paste a valid Azure OpenAI key and rerun this cell.")


## Step 2 - Configure Azure OpenAI client


In [24]:
endpoint = "https://gianm-m6osfju4-eastus2.openai.azure.com/"
model_name = "gpt-5.4"
deployment = "gpt-5.4-llm-kb-lessons"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

print("Azure OpenAI client configured")
print(f"Endpoint: {endpoint}")
print(f"Deployment: {deployment}")
print(f"API version: {api_version}")


Azure OpenAI client configured
Endpoint: https://gianm-m6osfju4-eastus2.openai.azure.com/
Deployment: gpt-5.4-llm-kb-lessons
API version: 2024-12-01-preview


## Step 3 - Test connection with a simple prompt


In [25]:
response = None
try:
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": "Hello! Briefly explain what Azure OpenAI Service is.",
            },
        ],
        max_completion_tokens=16384,
        temperature=0.7,
    )
    print("Model response:\n")
    print(response.choices[0].message.content)
except APIStatusError as exc:
    print(f"Connection test failed with HTTP {exc.status_code}.")
    print(f"Server message: {exc.message}")
    request_id = None
    if exc.response is not None:
        request_id = exc.response.headers.get("x-request-id") or exc.response.headers.get("x-ms-request-id")
    if request_id:
        print(f"Request ID: {request_id}")
    raise
except Exception as exc:
    print(f"Connection test failed: {type(exc).__name__}: {exc}")
    print("Check endpoint, deployment name, API version, and key.")
    raise


Model response:

Azure OpenAI Service is Microsoft’s cloud offering that gives you access to OpenAI models through the Azure platform.

In brief, it lets organizations:
- Use powerful AI models for tasks like chat, text generation, summarization, coding help, and embeddings
- Deploy and manage those models within Azure’s enterprise environment
- Benefit from Azure features such as security, compliance, governance, and scalability
- Integrate AI capabilities into apps and workflows using Azure tools and APIs

It’s essentially a way for businesses to use OpenAI models with Azure’s infrastructure, administration, and enterprise controls.


## Step 4 - Load a PubMed document from the project


In [26]:
pubmed_dir = Path("..") / "data" / "pubmed"
available_files = sorted(pubmed_dir.glob("*.txt"))

if not available_files:
    raise FileNotFoundError(f"No .txt files found in {pubmed_dir.resolve()}")

print("Available PubMed documents:")
for file_path in available_files:
    print(f"- {file_path.name}")

selected = input("Enter the full filename to load (including extension, e.g. 39549628.txt): ").strip()
if not selected:
    raise ValueError("You must provide a filename including extension.")

available_names = {file_path.name for file_path in available_files}
if selected not in available_names:
    raise FileNotFoundError("Invalid selection. Enter one of the listed filenames with extension.")

pubmed_path = pubmed_dir / selected

document_text = pubmed_path.read_text(encoding="utf-8")

print(f"Loaded document: {pubmed_path.name}")
print(f"Characters: {len(document_text)}")
print(document_text[:500])


Available PubMed documents:
- 39549628.txt
Loaded document: 39549628.txt
Characters: 2203
The molecular mechanisms of steroid hormone effects on cognitive function
Hai Duc Nguyen, Giang Huong Vu, Woong-Ki Kim 

Objective: There is a lack of information on the molecular mechanisms by which steroid hormones (testosterone, estrogen, and progesterone) regulate cognitive impairment. Thus, we aimed to identify the protective effects of steroid hormones on cognitive function.

Methods: We analyzed the literature on the molecular mechanisms, biological activities, physicochemical properties,


## Step 5 - Ask for a short summary of the loaded document


In [27]:
prompt = (
    "You are a biomedical research assistant. "
    "Read the following scientific article and provide a concise summary with "
    "objective, methods, key findings, and conclusions.\n\n"
    f"--- ARTICLE ---\n{document_text}\n--- END OF ARTICLE ---\n\n"
    "Summary:"
)

summary_response = None
try:
    summary_response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": "You are a biomedical research assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=1,
    )

    summary = summary_response.choices[0].message.content
    print("Document summary:\n")
    print(summary)
except Exception as exc:
    print(f"Summary request failed: {type(exc).__name__}: {exc}")
    print("Try a shorter prompt or verify deployment settings.")
    raise


Document summary:

**Objective:**  
To review and identify the molecular mechanisms by which steroid hormones—testosterone, estrogen, and progesterone—may protect against cognitive impairment.

**Methods:**  
The authors conducted a literature-based analysis focusing on the molecular mechanisms, biological activities, physicochemical properties, and pharmacokinetics of steroid hormones.

**Key Findings:**  
- Steroid hormones may protect cognitive function through regulation of key genes, especially **INS, TNF, STAT3, and ESR1**.  
- Important regulatory molecules involved include microRNAs **hsa-miR-335-5p, hsa-miR-16-5p, and hsa-miR-26b-5p**, and transcription factors such as **NFKB1, PPARG, NR3C1, GATA2, EGR1, ATF3, and CEBPA**.  
- Mechanistically, their effects are linked to **cognitive processing, phosphorylation regulation, neuronal apoptosis, and Alzheimer’s disease–related signaling pathways** within protein-protein interaction networks.  
- Steroid hormones also show **anti-i

## Step 6 - Show token usage


In [28]:
if summary_response is not None and summary_response.usage is not None:
    usage = summary_response.usage
    print(f"Prompt tokens    : {usage.prompt_tokens}")
    print(f"Completion tokens: {usage.completion_tokens}")
    print(f"Total tokens     : {usage.total_tokens}")
else:
    print("No token usage available because the summary call did not complete.")


Prompt tokens    : 511
Completion tokens: 441
Total tokens     : 952


## Step 7 - Load prompts and choose a PubMed document for the user prompt


In [29]:
prompt_dir = Path("..") / "data" / "prompt"
system_prompt_path = prompt_dir / "system_prompt_generic.txt"
user_prompt_path = prompt_dir / "user_prompt.txt"

if not prompt_dir.exists():
    raise FileNotFoundError(f"Prompt directory not found: {prompt_dir.resolve()}")
if not system_prompt_path.exists():
    raise FileNotFoundError(f"Missing file: {system_prompt_path.resolve()}")
if not user_prompt_path.exists():
    raise FileNotFoundError(f"Missing file: {user_prompt_path.resolve()}")

system_prompt_text = system_prompt_path.read_text(encoding="utf-8").strip()
user_prompt_text = user_prompt_path.read_text(encoding="utf-8").strip()

if not system_prompt_text:
    raise ValueError(f"System prompt file is empty: {system_prompt_path.resolve()}")
if not user_prompt_text:
    raise ValueError(f"User prompt file is empty: {user_prompt_path.resolve()}")

print(f"Loaded system prompt: {system_prompt_path.name} ({len(system_prompt_text)} chars)")
print(f"Loaded user prompt  : {user_prompt_path.name} ({len(user_prompt_text)} chars)")
print("\nSystem prompt preview:\n")
print(system_prompt_text[:250])
print("\nUser prompt preview:\n")
print(user_prompt_text[:250])

pubmed_dir = Path("..") / "data" / "pubmed"
available_files = sorted(pubmed_dir.glob("*.txt"))
if not available_files:
    raise FileNotFoundError(f"No .txt files found in {pubmed_dir.resolve()}")

print("Available PubMed documents for prompt injection:")
for file_path in available_files:
    print(f"- {file_path.name}")

choice = input("Enter the full filename to inject in the prompt (including extension): ").strip()
if not choice:
    raise ValueError("You must provide a filename including extension.")

available_names = {file_path.name for file_path in available_files}
if choice not in available_names:
    raise FileNotFoundError("Invalid selection. Enter one of the listed filenames with extension.")

selected_pubmed_path = pubmed_dir / choice
selected_document_text = selected_pubmed_path.read_text(encoding="utf-8")

print(f"\nUsing document: {selected_pubmed_path.name}")

if "{document_text}" in user_prompt_text:
    final_user_prompt = user_prompt_text.replace("{document_text}", selected_document_text)
else:
    print("Warning: `{document_text}` placeholder not found in user prompt. Appending document text.")
    final_user_prompt = (
        f"{user_prompt_text}\n\n"
        f"--- PUBMED DOCUMENT ({selected_pubmed_path.name}) ---\n"
        f"{selected_document_text}\n"
        "--- END PUBMED DOCUMENT ---"
    )

summary_response = None
try:
    summary_response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": system_prompt_text},
            {"role": "user", "content": final_user_prompt},
        ],
        temperature=1,
    )

    summary = summary_response.choices[0].message.content
    print("Document summary:\n")
    print(summary)
    if summary_response is not None and summary_response.usage is not None:
        usage = summary_response.usage
        print(f"Prompt tokens    : {usage.prompt_tokens}")
        print(f"Completion tokens: {usage.completion_tokens}")
        print(f"Total tokens     : {usage.total_tokens}")
    else:
        print("No token usage available because the summary call did not complete.")

except Exception as exc:
    print(f"Summary request failed: {type(exc).__name__}: {exc}")
    print("Try a shorter prompt or verify deployment settings.")
    raise



Loaded system prompt: system_prompt_generic.txt (1275 chars)
Loaded user prompt  : user_prompt.txt (86 chars)

System prompt preview:

Task = """
You are an expert biomedical information extraction system specialized in:
Knowledge graph construction from scientific text using named entity recognition and relation extraction.
The gut-brain axis / gut-brain interplay, including microb

User prompt preview:

Extract entities and relations from the following document.

Document:
{document_text}
Available PubMed documents for prompt injection:
- 39549628.txt

Using document: 39549628.txt
Document summary:

[Steroid hormones, CHEMICAL, REGULATE, cognitive impairment, DISEASE]
[testosterone, STEROID_HORMONE, REGULATE, cognitive impairment, DISEASE]
[estrogen, STEROID_HORMONE, REGULATE, cognitive impairment, DISEASE]
[progesterone, STEROID_HORMONE, REGULATE, cognitive impairment, DISEASE]
[Steroid hormones, CHEMICAL, PROTECT_AGAINST, cognitive impairment, DISEASE]
[Steroid hormones, CHEMICAL, 